# Emy v2 — Streamlit UI on Google Colab

Run the full Emy Streamlit chat interface from Google Colab using **localtunnel** to expose the app publicly.

This notebook:
1. Installs Ollama (architecture-detected) and pulls required models
2. Installs Emy and its dependencies
3. Launches the Streamlit app with a public URL via localtunnel

> **Note:** localtunnel gives you a temporary public URL. You'll need to click through a confirmation page on first visit.

## 1) Install Ollama

In [ ]:
import os, subprocess, time, platform

print("Machine:", platform.machine())
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Clean up any stale binary
    !rm -f /usr/local/bin/ollama ./ollama

    # Detect architecture
    arch = platform.machine().lower()
    if arch in ("x86_64", "amd64"):
        url = "https://ollama.com/download/ollama-linux-amd64.tar.zst"
    elif arch in ("aarch64", "arm64"):
        url = "https://ollama.com/download/ollama-linux-arm64.tar.zst"
    else:
        raise RuntimeError(f"Unsupported architecture: {arch}")

    !apt-get update -qq && apt-get install -y -qq zstd
    !wget -qO /tmp/ollama.tar.zst "$url"
    !tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /usr/local

    # Verify
    !/usr/local/bin/ollama --version
    print("Ollama installed successfully.")
else:
    print("Local environment detected — assuming Ollama is already installed.")
    print("If not, install from https://ollama.com")

## 2) Start Ollama server & pull models

In [ ]:
if IN_COLAB:
    proc = subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)
    print(f"Ollama server started (pid {proc.pid})")
else:
    print("Assuming Ollama is already running locally.")

# Pull models (cached after first run)
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text
print("Models ready.")

## 3) Install Emy + Streamlit dependencies

In [ ]:
# Install Emy from GitHub (includes streamlit, fastapi, faiss-cpu, etc.)
!pip install -q git+https://github.com/FallingObject/emy

# Install localtunnel via npm for public URL tunneling
if IN_COLAB:
    !npm install -g localtunnel 2>/dev/null
    print("localtunnel installed.")
else:
    print("Skipping localtunnel install (local environment).")

## 4) Quick sanity check

Verify Emy can connect to Ollama before launching the full UI.

In [ ]:
from pathlib import Path
from emy import Emy, EmyConfig

config = EmyConfig(workdir=Path("./runtime"), mode="train")
emy = Emy(config)

print("Ollama healthy:", emy.llm.health_check())
print("Model:", config.llm_model)
print("Embedding model:", config.embedding_model)
print("\nEmy is ready. Launching Streamlit next.")

## 5) (Optional) Ingest sample documents before launching UI

If you want Emy to have documents to search when the UI starts, ingest them now.

In [ ]:
# Uncomment to ingest sample docs:
# count = emy.ingest("./data/sample_docs")
# print(f"Ingested {count} chunks")

# Or upload your own files through the Streamlit UI once it launches.

## 6) Launch Streamlit + localtunnel

This cell:
1. Starts Streamlit on port **8501** in the background
2. Starts a **localtunnel** tunnel to expose it publicly
3. Prints the public URL you can open in any browser

> On first visit, localtunnel shows a confirmation page — just click through it.

**This cell runs until you stop it.** Use the stop button or `Runtime > Interrupt` to shut down.

In [ ]:
import subprocess, time, re, threading

STREAMLIT_PORT = 8501

# Find the streamlit_app.py path
import emy as _emy_pkg
app_path = Path(_emy_pkg.__file__).parent / "streamlit_app.py"
print(f"Streamlit app: {app_path}")

# Start Streamlit in background
st_proc = subprocess.Popen(
    [
        "streamlit", "run", str(app_path),
        "--server.port", str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(f"Streamlit started (pid {st_proc.pid}), waiting for it to be ready...")
time.sleep(5)

if IN_COLAB:
    # Start localtunnel and capture the public URL
    lt_proc = subprocess.Popen(
        ["lt", "--port", str(STREAMLIT_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    # Read the URL from localtunnel stdout
    url_line = lt_proc.stdout.readline().decode().strip()
    print("\n" + "=" * 60)
    print(f"  Emy Streamlit UI is live at:")
    print(f"  {url_line}")
    print("=" * 60)
    print("\nClick the URL above to open the Emy chat interface.")
    print("(You may need to click through a localtunnel confirmation page.)")
    print("\nPress the stop button to shut down.")

    # Keep the cell alive
    try:
        lt_proc.wait()
    except KeyboardInterrupt:
        print("\nShutting down...")
        lt_proc.terminate()
        st_proc.terminate()
else:
    print(f"\nStreamlit is running at: http://localhost:{STREAMLIT_PORT}")
    print("Open the URL above in your browser.")
    print("Press the stop button to shut down.")
    try:
        st_proc.wait()
    except KeyboardInterrupt:
        print("\nShutting down...")
        st_proc.terminate()

## 7) Cleanup

Run this cell after you're done to kill background processes.

In [ ]:
# Kill Streamlit and localtunnel if still running
import os, signal

for name, p in [("Streamlit", st_proc), ("localtunnel", lt_proc if IN_COLAB else None)]:
    if p and p.poll() is None:
        p.terminate()
        print(f"{name} stopped.")
    else:
        print(f"{name} already stopped.")

print("Done.")